# Esse notebook aplica regressão linear, avaliação de lucro e bootstraping para selecionar a melhor região 

In [1]:
# Importação das bibliotecas necessárias
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [2]:
# Carregamento dos datasets das três regiões

geo0 = pd.read_csv('/datasets/geo_data_0.csv')
geo1 = pd.read_csv('/datasets/geo_data_1.csv')
geo2 = pd.read_csv('/datasets/geo_data_2.csv')

In [3]:
# Verificação da estrutura dos datasets

for i, df in enumerate([geo0, geo1, geo2]):
    
    print(f'\nREGIÃO {i}')
    
    print(df.head())
    
    print(df.info())
    
    print(df.describe())
    
    print('Valores ausentes:')
    print(df.isna().sum())
    
    print('Duplicatas:', df.duplicated().sum())


REGIÃO 0
      id        f0        f1        f2     product
0  txEyH  0.705745 -0.497823  1.221170  105.280062
1  2acmU  1.334711 -0.340164  4.365080   73.037750
2  409Wp  1.022732  0.151990  1.419926   85.265647
3  iJLyR -0.032172  0.139033  2.978566  168.620776
4  Xdl7t  1.988431  0.155413  4.751769  154.036647
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None
                  f0             f1             f2        product
count  100000.000000  100000.000000  100000.000000  100000.000000
mean        0.500419       0.250143       2.502647      92.500000
std         0.871832       0.504433       3.248

Os três conjuntos de dados possuem a mesma estrutura, contendo um identificador do poço, três variáveis explicativas e a variável alvo product.

Não foram encontrados valores ausentes nem registros duplicados.

Dessa forma, não foi necessária nenhuma etapa adicional de limpeza dos dados.

In [4]:
def train_model(df):
    X = df[['f0','f1','f2']]
    y = df['product']
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42)
    model = LinearRegression()
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    print('RMSE:', mean_squared_error(y_val, preds)**0.5)
    print('Média prevista:', preds.mean())
    return y_val.reset_index(drop=True), pd.Series(preds)

In [5]:
# Condições de negócio fornecidas pela empresa

BUDGET = 100_000_000

REVENUE_PER_UNIT = 4500

SELECTED_WELLS = 200

EXPLORED_POINTS = 500

In [6]:
# Quantidade mínima de petróleo necessária para cobrir os custos

break_even = BUDGET / (SELECTED_WELLS * REVENUE_PER_UNIT)

print(break_even)

111.11111111111111


Considerando o orçamento de 100 milhões de dólares para o desenvolvimento de 200 poços e uma receita de 4.500 dólares por unidade produzida, cada poço precisa produzir aproximadamente 111,1 mil barris para atingir o ponto de equilíbrio.

In [7]:
def train_region(df):
    
    # Separação das variáveis explicativas e alvo
    
    features = df[['f0', 'f1', 'f2']]
    
    target = df['product']
    
    # Divisão em treino e validação
    
    features_train, features_valid, target_train, target_valid = (
        train_test_split(
            features,
            target,
            test_size=0.25,
            random_state=42
        )
    )
    
    # Treinamento do modelo
    
    model = LinearRegression()
    
    model.fit(features_train, target_train)
    
    # Previsões
    
    predictions = model.predict(features_valid)
    
    # Cálculo do RMSE
    
    rmse = mean_squared_error(
        target_valid,
        predictions,
        squared=False
    )
    
    print('Volume médio previsto:', predictions.mean())
    
    print('RMSE:', rmse)
    
    return (
        target_valid.reset_index(drop=True),
        pd.Series(predictions)
    )

In [8]:
# Região 0

target0, pred0 = train_region(geo0)

# Região 1

target1, pred1 = train_region(geo1)

# Região 2

target2, pred2 = train_region(geo2)

Volume médio previsto: 92.39879990657768
RMSE: 37.75660035026169
Volume médio previsto: 68.71287803913762
RMSE: 0.890280100102884
Volume médio previsto: 94.77102387765939
RMSE: 40.145872311342174


A Região X apresentou o menor RMSE, indicando maior precisão nas previsões. Entretanto, a escolha final dependerá não apenas da qualidade do modelo, mas também da análise econômica e do risco associado a cada região.

In [9]:
def calculate_profit(target, predictions):
    
    # Ordenação das previsões
    
    predictions = pd.Series(predictions)
    
    best_wells = predictions.sort_values(
        ascending=False
    ).head(200)
    
    # Seleção dos poços correspondentes
    
    selected_target = target.loc[
        best_wells.index
    ]
    
    # Receita
    
    revenue = (
        selected_target.sum()
        * REVENUE_PER_UNIT
    )
    
    # Lucro
    
    profit = revenue - BUDGET
    
    return profit

In [10]:
profit0 = calculate_profit(target0, pred0)

profit1 = calculate_profit(target1, pred1)

profit2 = calculate_profit(target2, pred2)

print(profit0)
print(profit1)
print(profit2)

33591411.14462179
24150866.966815114
25985717.59374112


Considerando apenas os 200 poços com maior volume previsto, a Região X apresentou o maior lucro potencial.

# Bootstrap

In [11]:
def bootstrap(target, predictions):
    
    state = np.random.RandomState(42)
    
    values = []
    
    for i in range(1000):
        
        target_subsample = target.sample(
            n=500,
            replace=True,
            random_state=state
        )
        
        predictions_subsample = predictions.loc[
            target_subsample.index
        ]
        
        profit = calculate_profit(
            target_subsample,
            predictions_subsample
        )
        
        values.append(profit)
    
    values = pd.Series(values)
    
    mean_profit = values.mean()
    
    lower = values.quantile(0.025)
    
    upper = values.quantile(0.975)
    
    risk = (
        values < 0
    ).mean() * 100
    
    return mean_profit, lower, upper, risk

# Aplicação do bootstrap

In [12]:
results = {}

for region, target, pred in [
    ('Região 0', target0, pred0),
    ('Região 1', target1, pred1),
    ('Região 2', target2, pred2)
]:
    
    results[region] = bootstrap(
        target,
        pred
    )

results

{'Região 0': (6061226.315266022, 100894.11995841737, 12463709.805645682, 2.5),
 'Região 1': (6651176.537521032, 1808515.8498991863, 12057104.608184986, 0.2),
 'Região 2': (5851036.3754606135, -8369.423215681249, 12120508.982598253, 2.6)}

# Tabela final

In [13]:

summary = pd.DataFrame(
    results,
    index=[
        'Lucro Médio',
        'IC Inferior',
        'IC Superior',
        'Risco (%)'
    ]
)

summary.T


,Lucro Médio,IC Inferior,IC Superior,Risco (%)
Região 0,6.061226e+06,1.008941e+05,1.246371e+07,2.5
Região 1,6.651177e+06,1.808516e+06,1.205710e+07,0.2
Região 2,5.851036e+06,-8.369423e+03,1.212051e+07,2.6


# Conclusão Final


O objetivo deste projeto foi identificar a região mais promissora para o desenvolvimento de novos poços de petróleo para a empresa OilyGiant. Para isso, foram utilizados dados geológicos de três regiões e construídos modelos de regressão linear capazes de prever o volume de reservas de petróleo em novos poços. Além da qualidade preditiva dos modelos, também foram considerados fatores econômicos e riscos financeiros, utilizando a técnica de bootstrapping para apoiar a tomada de decisão.

Inicialmente, foi realizada a análise dos dados das três regiões. Os conjuntos apresentaram estrutura consistente, sem valores ausentes ou duplicados, permitindo a utilização direta das informações para treinamento dos modelos. Em seguida, os dados foram divididos em conjuntos de treinamento e validação na proporção de 75% e 25%, conforme especificado nos requisitos do projeto.

Foram treinados três modelos de regressão linear, um para cada região. Os resultados de desempenho foram os seguintes:

 Região	/ Volume Médio Previsto/ RMSE
- Região 0	  / 92,40  / 37,76
- Região 1	 / 68,71 / 0,89
- Região 2	/ 94,77 /40,15

A Região 1 apresentou o menor RMSE entre todas as regiões, indicando uma capacidade de previsão significativamente superior. Isso demonstra que as previsões produzidas pelo modelo para essa região são muito mais precisas e confiáveis do que as obtidas para as demais regiões.

Na etapa de análise econômica, foi calculado o ponto de equilíbrio do investimento. Considerando um orçamento total de US$ 100 milhões para o desenvolvimento de 200 poços e uma receita de US$ 4.500 por unidade produzida, verificou-se que cada poço precisa produzir aproximadamente 111,1 mil barris para cobrir os custos do projeto. Os volumes médios previstos observados nas regiões ficaram abaixo desse valor, evidenciando que a seleção estratégica dos melhores poços é fundamental para garantir a rentabilidade do investimento.

Para cada região foram selecionados os 200 poços com maior potencial produtivo, de acordo com as previsões dos modelos. Em seguida, foi realizada uma análise de risco utilizando bootstrapping com 1.000 amostras para estimar a distribuição dos lucros e calcular métricas de rentabilidade e segurança financeira.

Os resultados obtidos foram:

Região / Lucro Médio (US$) /IC 95% Inferior /IC 95% Superior/Risco de Prejuízo
- Região 0	/ 6.061.226	/ 100.894	/ 12.463.710	/ 2,5%
- Região 1	/ 6.651.177	/ 1.808.516	/ 12.057.105	/ 0,2%
- Região 2	/ 5.851.036	/ -8.369	/ 12.120.509	/ 2,6%

A análise dos resultados demonstra que a Região 1 apresenta o melhor equilíbrio entre retorno financeiro e segurança operacional. Além de possuir o maior lucro médio esperado entre as três regiões, ela apresentou uma probabilidade de prejuízo de apenas 0,2%, muito abaixo do limite máximo aceitável de 2,5% estabelecido pela empresa.

A Região 0 apresentou lucro médio positivo e um intervalo de confiança totalmente positivo, porém seu risco de prejuízo ficou exatamente no limite permitido (2,5%), tornando-a uma opção menos segura quando comparada à Região 1.

Já a Região 2 apresentou o menor lucro médio e um risco de prejuízo de 2,6%, ultrapassando o limite máximo aceitável. Além disso, o limite inferior do seu intervalo de confiança é negativo, indicando a possibilidade de perdas financeiras em cenários desfavoráveis. Por esse motivo, essa região não deve ser considerada para investimento.

Portanto, após avaliar o desempenho dos modelos, o potencial de lucro e os riscos associados a cada alternativa, recomenda-se que a OilyGiant invista na Região 1. Essa região apresentou o maior lucro médio esperado, a menor probabilidade de prejuízo e o modelo mais preciso entre as três regiões analisadas, tornando-se a opção mais segura e economicamente vantajosa para o desenvolvimento de novos poços de petróleo.

**a Região 1 não possui o maior volume médio previsto, mas possui a melhor qualidade de previsão (RMSE extremamente baixo) e o menor risco financeiro.**